# Joint Kinematics: Cross-Subject Outlier Removal and Group Statistics

This notebook is **Stage 3** of my joint kinematics analysis pipeline. It performs cross-subject outlier removal on the joint angle peak/ROM measurements, then runs group-level statistical comparisons (Training vs Control × Time 1 vs Time 2) with publication-style boxplots and significance annotation.

**Author**: Yeon-Joo Kang | Georgia State University | 2021–2024
**Status**: Archived. Reflects my analytical approach during PhD dissertation research.

## Input dependencies

This notebook works in two phases with different input files:

**Phase A: Cross-subject outlier removal**
- `{Subject}_jangle_data.csv` — per-subject combined joint angle peak/ROM descriptive dataframe produced by Stage 1

**Phase B: Group-level statistics**
- `JointK.csv` — long-format dataframe with columns for Subject, Group (Training/Control), Time (1/2), and joint kinematic measurements

## Pipeline context

This is the **third and final stage** of the descriptive-statistics joint kinematics workflow:

1. `vicon_joint_kinematics_peak_rom.ipynb` — per-trial gait-cycle segmentation and per-trial outlier removal
2. `joint_kinematics_ensemble_avg.ipynb` — pool sides, compute per-subject ensemble curves
3. **This notebook** — cross-subject outlier removal, group comparison statistics

Time-series statistical comparison via SPM (Statistical Parametric Mapping) is handled separately and is being developed in another repository.

## Statistical methods

- **Outlier removal**: IQR rule applied column-wise (Q1−1.5×IQR / Q3+1.5×IQR), values outside the bounds are replaced with NaN rather than removing the row
- **Group comparison**: mixed-design ANOVA with Group as between-subjects factor and Time as within-subjects factor, computed with `pingouin`
- **Post-hoc**: pairwise t-tests via `pingouin.pairwise_tests`
- **Visualization**: seaborn boxplots with significance annotation via `statannotations`

---


## Phase A — Cross-Subject Outlier Removal

### 1. Setup and Data Import

Import the per-subject combined joint angle descriptive dataframe.

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
import os,sys
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Check current working directory
os.getcwd()

In [ ]:
# File import by filename

Subject = input("Subject: ")

df = pd.read_csv(Subject + '_jangle_data'+'.csv')
df = df.drop("Unnamed: 0", axis=1)

df.head()

Drop any rows where the first column matches the column header text (a Vicon export artifact where headers can repeat partway down the file).

In [ ]:
# Drop the rows with column names
first_word = df.columns[0]
rows_to_drop = [] 
for index, value in df.iloc[:, 0].items():  
    if first_word in str(value): 
        rows_to_drop.append(index)

df = df.drop(index=rows_to_drop)   
df=df.astype(float)
df

### 2. Build Combined Left + Right Dataframe

The original CSV stacks left and right side measurements side by side. To pool left and right cycles for outlier analysis, I separate them by column name pattern (`l*` for left, `r*` for right), then stack them as additional rows.

In [ ]:
Ldfd = df[['lamin', 'lamax', 'laROM', 'min_time', 'max_time', 
            'lhmin', 'lhmax', 'lhROM','min_time.2', 'max_time.2', 
            'lkmin', 'lkmax', 'lkROM', 'min_time.4', 'max_time.4']]

Ldfd.columns =['AnklePLantar', 'AnkleDorsi', 'AnkleROM', 'AnklePLantar%',
       'AnkleDorsix%', 'HipExt', 'HipFlex', 'HipROM', 'HipExt%', 'HipFlex%',
       'KneeExt', 'KneeFlex', 'KneeROM', 'KneeExt%', 'KneeFlex%']

Rdfd = df[['ramin', 'ramax','raROM', 'min_time.1', 'max_time.1', 
           'rhmin', 'rhmax', 'rhROM', 'min_time.3','max_time.3', 
           'rkmin', 'rkmax', 'rkROM', 'min_time.5', 'max_time.5']]

Rdfd.columns =['AnklePLantar', 'AnkleDorsi', 'AnkleROM', 'AnklePLantar%',
       'AnkleDorsix%', 'HipExt', 'HipFlex', 'HipROM', 'HipExt%', 'HipFlex%',
       'KneeExt', 'KneeFlex', 'KneeROM', 'KneeExt%', 'KneeFlex%']


In [ ]:
dfd = pd.concat([Ldfd, Rdfd], axis=0)
dfd = dfd.reset_index()
dfd = dfd.drop(['index'], axis=1)

In [ ]:
# Get column names to remove outliers

col_names = dfd.columns
col_names

### 3. Outlier Removal via IQR Rule

Define a column-wise IQR outlier detection function. Values outside Q1−1.5×IQR or Q3+1.5×IQR are replaced with NaN. Replacing with NaN (rather than dropping the row) preserves the dataframe size and keeps each subject's other measurements intact.

In [ ]:
# Define Outlier remove function

def detect_outliers(df,columns):
    q1=df[columns].quantile(0.25)
    q3=df[columns].quantile(0.75)
    iqr=q3-q1
    
    boundary=1.5*iqr
    
    index1=df[df[columns] > q3+boundary].index
    index2=df[df[columns] < q1-boundary].index 
    
    df[columns]=df[columns].drop(index1)
    df[columns]=df[columns].drop(index2)
    
    return df

In [ ]:
# Excute Outlier remove function

for col in col_names:
    rmd_dfd = detect_outliers(dfd, col)

rmd_dfd

In [ ]:
new_rmd_dfd = rmd_dfd

In [ ]:
# Replace zero to NaN for calculating Mean and SD

new_rmd_dfd.replace(0, np.nan, inplace=True)
new_rmd_dfd

### 4. Descriptive Statistics and Save

Compute the per-column mean and SD across all (subject × side) rows. These are the inputs to the group-level analysis in Phase B.

In [ ]:
# Data Characteristics
des = new_rmd_dfd.describe()
des

In [ ]:
# Get only Mean, SD, and Count

des_mean = des.loc['mean',:]
dfd_m = des_mean.to_frame().T
des_sd = des.loc['std',:]
dfd_sd = des_sd.to_frame().T

In [ ]:
# Save Meand and SD for outliser removed and normalized data in csv

if not os.path.exists('rom_data.csv'):
    dfd_m.to_csv('rom_mean.csv', index = Subject, mode='w')
    dfd_s.to_csv('rom_SD.csv', index = Subject, mode='w')
else:
    dfd_m.to_csv('rom_mean.csv', index = Subject, mode='a')
    dfd_s.to_csv('rom_SD.csv', index = Subject, mode='a')


### 5. Per-Column Boxplots

Visualize the distribution of each joint angle measure across all subjects after outlier removal. I checked these per-column rather than as a grid because at the time I wanted to inspect each measure individually.

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[0]]), sns.boxplot(y=rmd_dfd[dfd.columns[1]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[2]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[3]]), sns.boxplot(y=rmd_dfd[dfd.columns[4]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[5]]), sns.boxplot(y=rmd_dfd[dfd.columns[6]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[7]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[8]]), sns.boxplot(y=rmd_dfd[dfd.columns[9]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[10]]), sns.boxplot(y=rmd_dfd[dfd.columns[11]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[12]])

In [ ]:
sns.boxplot(y=rmd_dfd[dfd.columns[13]]), sns.boxplot(y=rmd_dfd[dfd.columns[14]])

## Phase B — Group-Level Statistics

### 6. Load Long-Format Group Data

For group comparison I work with a long-format CSV (`JointK.csv`) that has one row per subject × time point with columns for Group (Training/Control), Time (1/2), Subject, and the joint kinematic measurements. This is built outside this notebook from the cross-subject outputs of Phase A.

In [ ]:
# Secondary Analysis
df = pd.read_csv('JointK'+'.csv')

In [ ]:
df.loc[df['Time'] == 1, 'Time'] = 'T1'
df.loc[df['Time'] == 2, 'Time'] = 'T2'

In [ ]:
df.loc[df['Group'] == 'Training', 'Group'] = 'Experimental'

### 7. Group × Time Boxplots

Side-by-side boxplots split by Group on the X axis with Time as hue. Show how each joint measure compares between groups and between time points.

In [ ]:
sns.boxplot(data=new_df, x='Group', y='AnklePLantar', hue='Time')

In [ ]:
sns.boxplot(data=new_df, x='Group', y='AnkleDorsi', hue='Time')

In [ ]:
sns.boxplot(data=new_df, x='Group', y='AnkleROM', hue='Time')

### 8. Mixed ANOVA and Post-Hoc Tests

For each joint measure, run a mixed-design ANOVA (Group between-subjects × Time within-subjects) using `pingouin.mixed_anova`, then pairwise post-hoc tests via `pingouin.pairwise_tests`. Significance from the post-hoc results drives the annotation bars on the boxplots.

In [ ]:
# Run statistics to get p-value
import scipy
import pingouin as pg
import statannot
from statannotations.Annotator import Annotator  # New import

In [ ]:
anova_results = pg.mixed_anova(dv="HipExtTime", between="Group", within="Time", subject="Subject", data=df)
posthoc_results = pg.pairwise_tests(dv="HipExtTime", between="Group", within="Time", subject="Subject", data=df, padjust="bonf")
sig_comparisons = posthoc_results[
    (posthoc_results["A"].isin(["T1", "Experimental", "Control"])) & 
    (posthoc_results["B"].isin(["T2", "Experimental", "Control"])) & 
    (posthoc_results["p-unc"] < 0.05)
][["A", "B", "Time", "p-unc"]]

#if not sig_comparisons.empty:
    # Create box_pairs formatted as (Group, Time) comparisons
#    box_pairs = [((row["A"], row["Time"]), (row["B"], row["Time"])) for _, row in sig_comparisons.iterrows()]
box_pairs = [(("Experimental", "T1"), ("Control", "T1")), (("Experimental", "T2"), ("Control", "T2"))]
p_values = sig_comparisons["p-unc"].tolist()

plt.figure(figsize=(8, 6))
ax = sns.boxplot(data=df, x="Time", y="KneeExtTime", hue="Group")

if box_pairs:
    annotator = Annotator(ax, box_pairs, data=df, x="Time", y="HipExtTime", hue="Group")
    annotator.set_pvalues(p_values)  # Use precomputed p-values from post-hoc tests
    annotator.configure(text_format="star", loc="outside")  # Set annotation style
    annotator.apply_and_annotate()

### 9. Boxplot with Significance Annotations

Build a boxplot with significance bars overlaid. I went through several approaches here (statannotations, manual annotation, manual significance combination), eventually arriving at the working version below.

In [ ]:
# Create a boxplot
plt.figure(figsize=(8, 6))
ax = sns.boxplot(data=df, x="Group", y="HipExtTime", hue="Time")

# Define significant comparisons
box_pairs = [(("Experimental", "T1"), ("Control", "T1")), (("Experimental", "T2"), ("Control", "T2"))]

# Initialize the Annotator
annotator = Annotator(ax, box_pairs, data=df, x="Group", y="HipExtTime", hue="Time")

# Perform statistical tests and annotate
annotator.configure(test="t-test_ind", text_format="simple", loc="inside")
annotator.apply_and_annotate()



In [ ]:
plt.show()

Final annotation approach: a manually defined significance combination is rendered as a bar with asterisks above the boxes.

In [ ]:
def convert_pvalue_to_asterisks(p_time):
    if pvalue <= 0.0001:
        return "****"
    elif pvalue <= 0.001:
        return "***"
    elif pvalue <= 0.01:
        return "**"
    elif pvalue <= 0.05:
        return "*"
    return "ns"

In [ ]:
significant_combinations = [[(1, 2), 5.651011584290147e-18]]
significant_combinations

In [ ]:
sns.boxplot(data=df, x='Group', y='HipExtTime', hue='Time')
# Significance bars
for i, significant_combination in enumerate(significant_combinations):
    # Columns corresponding to the datasets of interest
    x1 = significant_combination[0][0]
    x2 = significant_combination[0][1]
    # What level is this bar among the bars above the plot?
    level = len(significant_combinations) - i
    # Plot the bar
    bar_height = (y_range * 0.07 * level) + top
    bar_tips = bar_height - (y_range * 0.02)
    plt.plot(
        [x1, x1, x2, x2],
        [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k'
    )
    # Significance level
    p = significant_combination[1]
    if p < 0.001:
        sig_symbol = '***'
    elif p < 0.01:
        sig_symbol = '**'
    elif p < 0.05:
        sig_symbol = '*'
    text_height = bar_height + (y_range * 0.01)
    plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', va='bottom', c='k')

plt.show()


In [ ]:
sns.boxplot(data=df, x='Time', y='KneeFlexTime', hue='Group')